# 09 · Explain H₂ energy / မော်လီကျူး စွမ်းအင်ကို လေ့လာခြင်း

**NexQus 1.3.0 lesson.** Install this checkout with the tutorial
extra: `python -m pip install -e ".[tutorial]"`, then restart the notebook kernel.
This fixed-geometry lesson works on native Windows and does not need PySCF.
To regenerate integrals and compare a bond-length curve against independent
PySCF FCI, use Linux, macOS or WSL on Windows, install the core first, then add
`python -m pip install -e ".[chemistry]"` and run `python examples/h2_vqe.py`.
PySCF does not support native Windows.

The coefficients below were generated with PySCF 2.14.0, RHF/STO-3G, neutral
singlet H₂ at 0.735 Å. They use Jordan–Wigner mapping and interleaved alpha/beta
spin orbitals in q0-first order. Total energies include nuclear repulsion.
The saved classical FCI reference is -1.1373060357534004 Hartree.

H₂ ကိုသာ အခြေခံထားသော ဥပမာဖြစ်ပြီး molecule အကြီးများ သို့မဟုတ် ဆေးဝါး၏
ထိရောက်မှုကို တွက်ချက်ထားခြင်း မဟုတ်ပါ။


In [ ]:
import sys
import nexqus
print('Kernel Python:', sys.executable)
print('Imported NexQus:', nexqus.__file__)
required = ('QuantumDistribution', 'QGAN', 'distribution_metrics', 'PauliHamiltonian')
missing = [name for name in required if not hasattr(nexqus, name)]
if missing:
    raise ImportError(
        'This kernel imported an older NexQus package without: ' + ', '.join(missing) + '. '
        'Install the current source into this notebook kernel with %pip install --upgrade '
        '"nexqus-sdk[tutorial] @ https://github.com/Inn0m1n4te030/NexQus-SDK/archive/refs/heads/main.zip" '
        'or %pip install -e "/path/to/NexQus-SDK[tutorial]", then restart the kernel and Run All. '
        'The published 1.2.0 wheel lacks these additions; install version 1.3.0 and restart. '
        'QGAN and this fixed-H2 lesson do not require the chemistry extra or PySCF.')


In [ ]:
import nexqus
if not all(hasattr(nexqus, name) for name in ('Molecule', 'molecule', 'molecular_gallery')):
    raise ImportError('Install the updated source checkout with python -m pip install -e ".[tutorial]", '
                      'then restart the notebook kernel. The molecular viewer is a new source addition. '
                      'Imported package: ' + str(nexqus.__file__))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import nexqus
from nexqus import Circuit, Simulator, Parameter, Noise, NoiseModel, algorithms
assert nexqus.__version__ == '1.3.0', 'Install NexQus SDK 1.3.0 in this kernel environment and restart it'
print('NexQus SDK:', nexqus.__version__)


## See the molecule first

This is the H₂ structure whose energy we calculate below: two hydrogen atoms,
0.735 Å apart. Drag to rotate. Select both atoms to measure the bond distance.
The spheres indicate atom positions; they are not electron-density clouds.

မော်လီကျူးကို လှည့်ကြည့်နိုင်ပြီး atom နှစ်ခုကို ရွေးကာ အကွာအဝေးကို တိုင်းနိုင်ပါသည်။


In [ ]:
from IPython.display import display
from nexqus import molecule
display(molecule('H2'))


In [ ]:
from nexqus import PauliHamiltonian, Adam
from nexqus.chemistry import h2_pair_ansatz, h2_reference_problem
problem = h2_reference_problem()  # the documented fixed-geometry coefficients
hamiltonian = problem.hamiltonian
ansatz = h2_pair_ansatz(Parameter('theta'))
energy = hamiltonian.function(ansatz)
parameters = {'theta': 0.}; optimizer = Adam(.04); energies = []
hf_energy = energy(parameters)
for epoch in range(180):
    value, gradient = energy.value_and_grad(parameters)
    energies.append(value)
    parameters = optimizer.step(parameters, gradient)
vqe_energy = energy(parameters)
fci_energy = -1.1373060357534004
print('Classical Hartree–Fock:', hf_energy)
print('Classical FCI reference:', fci_energy)
print('NexQus VQE:', vqe_energy)
assert abs(hf_energy - (-1.116998996754004)) < 1e-8
assert abs(vqe_energy-fci_energy) < 2e-7
display(problem.molecule().view(energies_hartree={
    'Classical Hartree–Fock': hf_energy, 'Classical FCI': fci_energy, 'NexQus VQE': vqe_energy},
    calculation_note='Fixed H2 at 0.735 Å, STO-3G, two electrons and four qubits. '
                     'VQE calculated above; total energies include nuclear repulsion.'))


## What is learned, and where does the energy come from?

The ansatz output is `cos(theta/2)|1100> + sin(theta/2)|0011>`.
It mixes a doubly occupied bonding orbital with a doubly excited determinant.
Every final state contains two electrons. This is a specialized H₂ family,
not a general-purpose chemistry ansatz.

The gate decomposition can temporarily leave the two-electron sector.
Intermediate Bloch states are circuit internals, not a molecular reaction
path. The explorer's `ZZII` readout is a single Pauli observable, not the
whole molecular energy. The plot below explicitly sums all energy terms.


In [ ]:
result = Simulator().run(ansatz.bind(parameters), shots=0, retain_state=True)
contributions = [c*result.expectation(word) for c, word in hamiltonian.terms]
assert abs(sum(contributions)-vqe_energy) < 1e-10
fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout='constrained')
axes[0].plot(energies, label='NexQus VQE')
axes[0].axhline(fci_energy, linestyle='--', label='Classical FCI')
axes[0].set(xlabel='Update', ylabel='Total energy / Hartree'); axes[0].legend()
axes[1].barh([word for _, word in hamiltonian.terms], contributions)
axes[1].set_xlabel('Energy contribution / Hartree')
plt.show()
from IPython.display import display
display(ansatz.explain(bindings=parameters, observable='ZZII',
    parameter_roles={'theta': 'weight'}, title='H₂ · determinant mixing'))
